In [19]:
%pip install neo4j mysql-connector-python pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [20]:
from getpass import getpass
from decimal import Decimal
from pathlib import Path
from datetime import date, datetime, timezone
import json
import re

try:
    import numpy as np
    import pandas as pd
    import mysql.connector
    from neo4j import GraphDatabase
    from IPython.display import display
except ImportError as error:
    raise SystemExit(
        "A required package is missing. Run: "
        "pip install neo4j mysql-connector-python pandas numpy"
    ) from error

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

mysql_host = (
    input("MySQL Host [localhost]: ").strip()
    or "localhost"
)

mysql_port_text = input(
    "MySQL Port [3306]: "
).strip()

mysql_port = (
    int(mysql_port_text)
    if mysql_port_text
    else 3306
)

mysql_user = (
    input("MySQL User [root]: ").strip()
    or "root"
)

mysql_database = (
    input(
        "MySQL Database [SchoolPulsePortfolio]: "
    ).strip()
    or "SchoolPulsePortfolio"
)

mysql_password = None

neo4j_uri = (
    input(
        "Neo4j URI [bolt://localhost:7687]: "
    ).strip()
    or "bolt://localhost:7687"
)

neo4j_user = (
    input("Neo4j User [neo4j]: ").strip()
    or "neo4j"
)

neo4j_database = (
    input("Neo4j Database [schoolpulse]: ").strip()
    or "schoolpulse"
)

neo4j_password = None

replace_existing_graph = False
batch_size = 1000

pipeline_name = "SchoolPulse Neo4j Pipeline Version 2"

pipeline_run_id = datetime.now(
    timezone.utc
).strftime(
    "RUN%Y%m%dT%H%M%SZ"
)

output_directory = (
    Path.cwd()
    / "SchoolPulse Portfolio Outputs"
)

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

print("Neo4j Pipeline Configuration Ready")

MySQL Host [localhost]:  
MySQL Port [3306]:  
MySQL User [root]:  
MySQL Database [SchoolPulsePortfolio]:  
Neo4j URI [bolt://localhost:7687]:  
Neo4j User [neo4j]:  
Neo4j Database [schoolpulse]:  


Neo4j Pipeline Configuration Ready


In [21]:
def pascal_to_snake(value):
    first_pass = re.sub(
        r"(.)([A-Z][a-z]+)",
        r"\1_\2",
        str(value),
    )

    second_pass = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        first_pass,
    )

    return second_pass.lower()


def clean_scalar(value):
    if value is None:
        return None

    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, Decimal):
        if value == value.to_integral_value():
            return int(value)

        return float(value)

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, datetime):
        return value.isoformat()

    if isinstance(value, date):
        return value.isoformat()

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
            list,
            tuple,
            dict,
        ),
    ):
        return value

    return str(value)


def clean_record(record):
    cleaned = {}

    for key, value in record.items():
        cleaned_value = clean_scalar(value)

        if cleaned_value is None:
            continue

        cleaned[key] = cleaned_value

    return cleaned


def clean_records(dataframe):
    return [
        clean_record(record)
        for record in dataframe.to_dict(
            orient="records"
        )
    ]


def professional_label(value):
    acronym_map = {
        "id": "ID",
        "ca": "CA",
        "sql": "SQL",
        "bi": "BI",
    }

    words = str(value).split("_")
    output_words = []

    for word in words:
        lower_word = word.lower()

        if lower_word in acronym_map:
            output_words.append(
                acronym_map[lower_word]
            )
        else:
            output_words.append(
                word.capitalize()
            )

    return " ".join(output_words)


def professional_frame(dataframe):
    result = dataframe.copy()

    result.columns = [
        professional_label(column)
        for column in result.columns
    ]

    return result


def show_table(title, dataframe, rows=20):
    print(title)
    display(
        professional_frame(
            dataframe.head(rows)
        )
    )


print("Utility Functions Ready")

Utility Functions Ready


In [22]:
mysql_connection = None

while mysql_connection is None:
    mysql_password = getpass(
        "Enter MySQL Password For "
        + mysql_user
        + ": "
    )

    try:
        mysql_connection = mysql.connector.connect(
            host=mysql_host,
            port=mysql_port,
            user=mysql_user,
            password=mysql_password,
            database=mysql_database,
            connection_timeout=10,
        )

    except mysql.connector.Error as error:
        if error.errno == 1045:
            print(
                "MySQL Login Was Not Accepted. "
                "Enter The Same Username And Password "
                "Used In MySQL Workbench.",
                flush=True,
            )

            mysql_connection = None
            continue

        if error.errno == 1049:
            raise RuntimeError(
                "The Database "
                + mysql_database
                + " Was Not Found. "
                "Run The Professional SQL Script First."
            ) from error

        if error.errno in {
            2003,
            2005,
        }:
            raise RuntimeError(
                "MySQL Could Not Be Reached At "
                + mysql_host
                + ":"
                + str(mysql_port)
                + ". Confirm That The MySQL Server Is Running."
            ) from error

        raise


def read_mysql_table(table_name):
    approved_tables = {
        "ClassLevel",
        "AcademicStream",
        "SchoolClass",
        "Student",
        "Teacher",
        "SchoolSubject",
        "StudentEnrollment",
        "ClassSubject",
        "TeacherAssignment",
        "StudentMonthlySummary",
        "StudentRiskProfile",
        "StudentSubjectRisk",
    }

    if table_name not in approved_tables:
        raise ValueError(
            "Table Is Not Approved For This Pipeline"
        )

    cursor = mysql_connection.cursor(
        dictionary=True
    )

    try:
        cursor.execute(
            f"SELECT * FROM `{table_name}`"
        )

        rows = cursor.fetchall()

    finally:
        cursor.close()

    dataframe = pd.DataFrame(rows)

    if not dataframe.empty:
        dataframe.columns = [
            pascal_to_snake(column)
            for column in dataframe.columns
        ]

    return dataframe


source_tables = {
    table_name: read_mysql_table(
        table_name
    )
    for table_name in [
        "ClassLevel",
        "AcademicStream",
        "SchoolClass",
        "Student",
        "Teacher",
        "SchoolSubject",
        "StudentEnrollment",
        "ClassSubject",
        "TeacherAssignment",
        "StudentMonthlySummary",
        "StudentRiskProfile",
        "StudentSubjectRisk",
    ]
}

mysql_connection.close()

source_counts = pd.DataFrame(
    [
        {
            "source_table": table_name,
            "rows_loaded": len(dataframe),
        }
        for table_name, dataframe
        in source_tables.items()
    ]
)

show_table(
    "MySQL Source Data Loaded",
    source_counts,
    rows=len(source_counts),
)

Enter MySQL Password For root:  ········


MySQL Source Data Loaded


,Source Table,Rows Loaded
0,ClassLevel,6
1,AcademicStream,3
2,SchoolClass,18
3,Student,216
4,Teacher,45
5,SchoolSubject,35
6,StudentEnrollment,216
7,ClassSubject,236
8,TeacherAssignment,236
9,StudentMonthlySummary,864


In [23]:
class_level = source_tables[
    "ClassLevel"
]

academic_stream = source_tables[
    "AcademicStream"
]

school_class = source_tables[
    "SchoolClass"
]

student = source_tables[
    "Student"
]

teacher = source_tables[
    "Teacher"
]

school_subject = source_tables[
    "SchoolSubject"
]

student_enrollment = source_tables[
    "StudentEnrollment"
]

class_subject = source_tables[
    "ClassSubject"
]

teacher_assignment = source_tables[
    "TeacherAssignment"
]

student_monthly_summary = source_tables[
    "StudentMonthlySummary"
]

student_risk_profile = source_tables[
    "StudentRiskProfile"
]

student_subject_risk = source_tables[
    "StudentSubjectRisk"
]

class_nodes = (
    school_class.merge(
        class_level[
            [
                "class_level_id",
                "level_name",
                "school_section",
                "level_order",
            ]
        ],
        on="class_level_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        academic_stream[
            [
                "stream_id",
                "stream_name",
            ]
        ],
        on="stream_id",
        how="left",
        validate="many_to_one",
    )
)

class_nodes[
    "academic_group"
] = (
    class_nodes[
        "stream_name"
    ]
    .fillna(
        "Junior Secondary"
    )
)

student_nodes = student.copy()

student_nodes[
    "student_name"
] = (
    student_nodes[
        "first_name"
    ]
    + " "
    + student_nodes[
        "last_name"
    ]
)

teacher_nodes = teacher.copy()

teacher_nodes[
    "teacher_name"
] = (
    teacher_nodes[
        "first_name"
    ]
    + " "
    + teacher_nodes[
        "last_name"
    ]
)

enrollment_edges = (
    student_enrollment.merge(
        student_nodes[
            [
                "student_id",
                "student_name",
            ]
        ],
        on="student_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        class_nodes[
            [
                "class_id",
                "class_name",
                "level_name",
                "academic_group",
            ]
        ],
        on="class_id",
        how="left",
        validate="many_to_one",
    )
)

monthly_nodes = (
    student_monthly_summary.merge(
        student_risk_profile,
        on="monthly_summary_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        enrollment_edges[
            [
                "enrollment_id",
                "student_id",
                "class_id",
            ]
        ],
        on="enrollment_id",
        how="left",
        validate="many_to_one",
    )
)

subject_risk_nodes = (
    student_subject_risk.merge(
        monthly_nodes[
            [
                "monthly_summary_id",
                "student_id",
                "month_name",
            ]
        ],
        on="monthly_summary_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        class_subject[
            [
                "class_subject_id",
                "subject_id",
                "class_id",
            ]
        ],
        on="class_subject_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        school_subject[
            [
                "subject_id",
                "subject_name",
                "subject_category",
            ]
        ],
        on="subject_id",
        how="left",
        validate="many_to_one",
    )
)

guardian_nodes = student_nodes[
    [
        "student_id",
        "student_name",
    ]
].copy()

guardian_nodes[
    "guardian_id"
] = (
    "GUA"
    + guardian_nodes[
        "student_id"
    ].str.replace(
        "STU",
        "",
        regex=False,
    )
)

guardian_nodes[
    "guardian_name"
] = (
    "Guardian of "
    + guardian_nodes[
        "student_name"
    ]
)

guardian_nodes[
    "relationship_to_student"
] = "Parent or Guardian"

guardian_nodes[
    "contact_masked"
] = (
    "+234***"
    + guardian_nodes[
        "student_id"
    ].str[-4:]
)

high_subject_risks = (
    subject_risk_nodes[
        subject_risk_nodes[
            "risk_level"
        ] == "High"
    ]
    .sort_values(
        [
            "monthly_summary_id",
            "performance_gap",
            "subject_average",
        ],
        ascending=[
            True,
            True,
            True,
        ],
    )
    .drop_duplicates(
        subset=[
            "monthly_summary_id",
        ]
    )
    .copy()
)

intervention_nodes = high_subject_risks[
    [
        "subject_risk_id",
        "student_id",
        "monthly_summary_id",
        "subject_id",
        "subject_name",
        "month_name",
        "recommended_action",
        "risk_reason",
    ]
].copy()

intervention_nodes[
    "intervention_id"
] = (
    "INT"
    + intervention_nodes[
        "subject_risk_id"
    ]
)

intervention_nodes[
    "intervention_type"
] = "Targeted Academic Support"

intervention_nodes[
    "action_taken"
] = intervention_nodes[
    "recommended_action"
]

intervention_nodes[
    "intervention_status"
] = "Planned"

intervention_nodes[
    "created_at"
] = (
    "2026-"
    + intervention_nodes[
        "month_name"
    ].map(
        {
            "January": "01-31",
            "February": "02-28",
            "March": "03-31",
            "April": "04-30",
        }
    )
    + "T18:30:00Z"
)

class_subject_nodes = (
    class_subject.merge(
        class_nodes[
            [
                "class_id",
                "class_name",
                "academic_group",
            ]
        ],
        on="class_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        school_subject[
            [
                "subject_id",
                "subject_name",
                "subject_category",
            ]
        ],
        on="subject_id",
        how="left",
        validate="many_to_one",
    )
)

student_nodes["name"] = (
    student_nodes["student_name"]
)

student_nodes["display_name"] = (
    student_nodes["student_name"]
)

student_nodes["display_type"] = "Student"

student_nodes["display_detail"] = (
    "Student ID: "
    + student_nodes["student_id"]
)

class_nodes["name"] = (
    class_nodes["class_name"]
    + " | "
    + class_nodes["academic_group"]
)

class_nodes["display_name"] = (
    class_nodes["name"]
)

class_nodes["display_type"] = "Class"

class_nodes["display_detail"] = (
    class_nodes["level_name"]
    + " | "
    + class_nodes["academic_group"]
)

teacher_assignment_details = (
    teacher_assignment.merge(
        class_subject_nodes[
            [
                "class_subject_id",
                "subject_name",
                "class_name",
            ]
        ],
        on="class_subject_id",
        how="left",
        validate="many_to_one",
    )
)

teacher_profile = (
    teacher_assignment_details.groupby(
        "teacher_id",
        as_index=False,
    )
    .agg(
        subject_count=(
            "subject_name",
            "nunique",
        ),
        class_count=(
            "class_name",
            "nunique",
        ),
        primary_subject=(
            "subject_name",
            lambda values: (
                sorted(
                    {
                        str(value)
                        for value in values.dropna()
                    }
                )[0]
                if len(
                    values.dropna()
                )
                else "Teaching Staff"
            ),
        ),
    )
)

teacher_nodes = teacher_nodes.merge(
    teacher_profile,
    on="teacher_id",
    how="left",
    validate="one_to_one",
)

teacher_nodes["subject_count"] = (
    teacher_nodes["subject_count"]
    .fillna(0)
    .astype(int)
)

teacher_nodes["class_count"] = (
    teacher_nodes["class_count"]
    .fillna(0)
    .astype(int)
)

teacher_nodes["primary_subject"] = (
    teacher_nodes["primary_subject"]
    .fillna("Teaching Staff")
)

teacher_nodes["name"] = (
    teacher_nodes["teacher_name"]
)

teacher_nodes["display_name"] = (
    teacher_nodes["teacher_name"]
)

teacher_nodes["display_type"] = "Teacher"

teacher_nodes["display_detail"] = np.where(
    teacher_nodes["subject_count"] == 0,
    "Teaching Staff",
    np.where(
        teacher_nodes["subject_count"] == 1,
        (
            teacher_nodes["primary_subject"]
            + " | "
            + teacher_nodes["class_count"]
            .astype(str)
            + " Classes"
        ),
        (
            teacher_nodes["subject_count"]
            .astype(str)
            + " Subjects | "
            + teacher_nodes["class_count"]
            .astype(str)
            + " Classes"
        ),
    ),
)

school_subject["name"] = (
    school_subject["subject_name"]
)

school_subject["display_name"] = (
    school_subject["subject_name"]
)

school_subject["display_type"] = "Subject"

school_subject["display_detail"] = (
    school_subject["subject_category"]
)

class_subject_nodes["name"] = (
    class_subject_nodes["subject_name"]
    + " | "
    + class_subject_nodes["class_name"]
)

class_subject_nodes["display_name"] = (
    class_subject_nodes["name"]
)

class_subject_nodes["display_type"] = (
    "Class Subject"
)

class_subject_nodes["display_detail"] = (
    class_subject_nodes["academic_group"]
)

monthly_nodes["name"] = (
    monthly_nodes["month_name"]
    + " Summary | "
    + monthly_nodes["risk_level"]
    + " Risk"
)

monthly_nodes["display_name"] = (
    monthly_nodes["name"]
)

monthly_nodes["display_type"] = (
    "Monthly Summary"
)

monthly_nodes["display_detail"] = (
    "Score "
    + monthly_nodes["average_score"]
    .round(2)
    .astype(str)
    + " | Attendance "
    + monthly_nodes["attendance_percentage"]
    .round(2)
    .astype(str)
    + "%"
)

subject_risk_nodes["name"] = (
    subject_risk_nodes["subject_name"]
    + " | "
    + subject_risk_nodes["month_name"]
    + " | "
    + subject_risk_nodes["risk_level"]
    + " Risk"
)

subject_risk_nodes["display_name"] = (
    subject_risk_nodes["name"]
)

subject_risk_nodes["display_type"] = (
    "Subject Risk"
)

subject_risk_nodes["display_detail"] = (
    "Student Score "
    + subject_risk_nodes["subject_average"]
    .round(2)
    .astype(str)
    + " | Class Average "
    + subject_risk_nodes["class_subject_average"]
    .round(2)
    .astype(str)
)

guardian_nodes["name"] = (
    guardian_nodes["guardian_name"]
)

guardian_nodes["display_name"] = (
    guardian_nodes["guardian_name"]
)

guardian_nodes["display_type"] = "Guardian"

guardian_nodes["display_detail"] = (
    guardian_nodes["relationship_to_student"]
)

intervention_nodes["name"] = (
    "Support for "
    + intervention_nodes["subject_name"]
    + " | "
    + intervention_nodes["month_name"]
)

intervention_nodes["display_name"] = (
    intervention_nodes["name"]
)

intervention_nodes["display_type"] = (
    "Intervention"
)

intervention_nodes["display_detail"] = (
    intervention_nodes["intervention_status"]
)

april_performance = (
    subject_risk_nodes[
        subject_risk_nodes[
            "month_name"
        ] == "April"
    ]
    [
        [
            "student_id",
            "class_subject_id",
            "subject_average",
            "risk_level",
        ]
    ]
    .copy()
)

prepared_counts = pd.DataFrame(
    [
        {
            "graph_entity": "Students",
            "records": len(student_nodes),
        },
        {
            "graph_entity": "Classes",
            "records": len(class_nodes),
        },
        {
            "graph_entity": "Subjects",
            "records": len(school_subject),
        },
        {
            "graph_entity": "Teachers",
            "records": len(teacher_nodes),
        },
        {
            "graph_entity": "Class Subjects",
            "records": len(class_subject_nodes),
        },
        {
            "graph_entity": "Monthly Summaries",
            "records": len(monthly_nodes),
        },
        {
            "graph_entity": "Subject Risks",
            "records": len(subject_risk_nodes),
        },
        {
            "graph_entity": "Guardians",
            "records": len(guardian_nodes),
        },
        {
            "graph_entity": "Interventions",
            "records": len(intervention_nodes),
        },
    ]
)

show_table(
    "Prepared Graph Data",
    prepared_counts,
    rows=len(prepared_counts),
)

Prepared Graph Data


,Graph Entity,Records
0,Students,216
1,Classes,18
2,Subjects,35
3,Teachers,45
4,Class Subjects,236
5,Monthly Summaries,864
6,Subject Risks,11328
7,Guardians,216
8,Interventions,253


In [24]:
driver = None

while driver is None:
    neo4j_password = getpass(
        "Enter Neo4j Password For "
        + neo4j_user
        + ": "
    )

    candidate_driver = GraphDatabase.driver(
        neo4j_uri,
        auth=(
            neo4j_user,
            neo4j_password,
        ),
    )

    try:
        candidate_driver.verify_connectivity()
        driver = candidate_driver

    except Exception as error:
        candidate_driver.close()

        error_name = type(error).__name__

        if error_name == "AuthError":
            print(
                "Neo4j Login Was Not Accepted. "
                "Enter The Password For The Neo4j Database User.",
                flush=True,
            )

            driver = None
            continue

        raise


def run_query(
    query,
    parameters=None,
):
    with driver.session(
        database=neo4j_database
    ) as session:
        result = session.run(
            query,
            parameters or {},
        )

        return [
            record.data()
            for record in result
        ]


def run_batch(
    query,
    records,
    label,
):
    total_records = len(records)

    print(
        "Writing",
        label,
        ":",
        total_records,
        flush=True,
    )

    for start in range(
        0,
        total_records,
        batch_size,
    ):
        batch = records[
            start:
            start + batch_size
        ]

        run_query(
            query,
            {
                "rows": batch,
                "pipeline_run_id": pipeline_run_id,
            },
        )

        print(
            label,
            "Written:",
            min(
                start + batch_size,
                total_records,
            ),
            "Of",
            total_records,
            flush=True,
        )


server_details = pd.DataFrame(
    run_query(
        '''
        CALL dbms.components()
        YIELD name, versions, edition
        RETURN
            name,
            versions[0] AS version,
            edition
        '''
    )
)

show_table(
    "Neo4j Connection Successful",
    server_details,
    rows=len(server_details),
)

Enter Neo4j Password For neo4j:  ········


Neo4j Connection Successful


,Name,Version,Edition
0,Neo4j Kernel,2026.05.0,enterprise
1,Cypher,5,


In [25]:
managed_labels = {
    "Student",
    "Class",
    "Subject",
    "Teacher",
    "ClassSubject",
    "MonthlySummary",
    "SubjectRisk",
    "Guardian",
    "Intervention",
}

if replace_existing_graph:
    print(
        "Removing Existing SchoolPulse Graph",
        flush=True,
    )

    run_query(
        "MATCH (node) DETACH DELETE node"
    )

    existing_constraints = run_query(
        '''
        SHOW CONSTRAINTS
        YIELD
            name,
            entityType,
            labelsOrTypes
        RETURN
            name,
            entityType,
            labelsOrTypes
        '''
    )

    constraints_removed = []

    for constraint in existing_constraints:
        if constraint.get("entityType") != "NODE":
            continue

        labels = set(
            constraint.get(
                "labelsOrTypes"
            )
            or []
        )

        if not labels.intersection(
            managed_labels
        ):
            continue

        constraint_name = constraint[
            "name"
        ]

        safe_constraint_name = (
            constraint_name.replace(
                "`",
                "``",
            )
        )

        run_query(
            "DROP CONSTRAINT `"
            + safe_constraint_name
            + "` IF EXISTS"
        )

        constraints_removed.append(
            {
                "constraint_name": constraint_name,
                "constraint_action": (
                    "Removed Before Version 2 Rebuild"
                ),
            }
        )

    if constraints_removed:
        show_table(
            "Existing Neo4j Constraints Removed",
            pd.DataFrame(
                constraints_removed
            ),
            rows=len(
                constraints_removed
            ),
        )

constraints = [
    (
        "student_id_unique",
        "Student",
        "student_id",
    ),
    (
        "class_id_unique",
        "Class",
        "class_id",
    ),
    (
        "subject_id_unique",
        "Subject",
        "subject_id",
    ),
    (
        "teacher_id_unique",
        "Teacher",
        "teacher_id",
    ),
    (
        "class_subject_id_unique",
        "ClassSubject",
        "class_subject_id",
    ),
    (
        "monthly_summary_id_unique",
        "MonthlySummary",
        "monthly_summary_id",
    ),
    (
        "subject_risk_id_unique",
        "SubjectRisk",
        "subject_risk_id",
    ),
    (
        "guardian_id_unique",
        "Guardian",
        "guardian_id",
    ),
    (
        "intervention_id_unique",
        "Intervention",
        "intervention_id",
    ),
]

constraint_rows = []

for (
    constraint_name,
    label_name,
    property_name,
) in constraints:
    run_query(
        f'''
        CREATE CONSTRAINT {constraint_name}
        IF NOT EXISTS
        FOR (node:{label_name})
        REQUIRE node.{property_name} IS UNIQUE
        '''
    )

    constraint_rows.append(
        {
            "constraint_name": constraint_name,
            "label_name": label_name,
            "property_name": property_name,
            "constraint_action": (
                "Version 2 Constraint Ready"
            ),
        }
    )

constraint_report = pd.DataFrame(
    constraint_rows
)

show_table(
    "Neo4j Version 2 Constraint Validation",
    constraint_report,
    rows=len(
        constraint_report
    ),
)

print(
    "Neo4j Constraints Ready",
    flush=True,
)

Neo4j Version 2 Constraint Validation


,Constraint Name,Label Name,Property Name,Constraint Action
0,student_id_unique,Student,student_id,Version 2 Constraint Ready
1,class_id_unique,Class,class_id,Version 2 Constraint Ready
2,subject_id_unique,Subject,subject_id,Version 2 Constraint Ready
3,teacher_id_unique,Teacher,teacher_id,Version 2 Constraint Ready
4,class_subject_id_unique,ClassSubject,class_subject_id,Version 2 Constraint Ready
5,monthly_summary_id_unique,MonthlySummary,monthly_summary_id,Version 2 Constraint Ready
6,subject_risk_id_unique,SubjectRisk,subject_risk_id,Version 2 Constraint Ready
7,guardian_id_unique,Guardian,guardian_id,Version 2 Constraint Ready
8,intervention_id_unique,Intervention,intervention_id,Version 2 Constraint Ready


Neo4j Constraints Ready


In [26]:
run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Student {
        student_id: row.student_id
    })
    SET
        node.student_name = row.student_name,
        node.first_name = row.first_name,
        node.last_name = row.last_name,
        node.gender = row.gender,
        node.date_of_birth = row.date_of_birth,
        node.admission_date = row.admission_date,
        node.admission_status = row.admission_status,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        student_nodes
    ),
    "Student Nodes",
)

Writing Student Nodes : 216
Student Nodes Written: 216 Of 216


In [27]:
run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Class {
        class_id: row.class_id
    })
    SET
        node.class_name = row.class_name,
        node.class_section = row.class_section,
        node.level_name = row.level_name,
        node.school_section = row.school_section,
        node.level_order = row.level_order,
        node.academic_group = row.academic_group,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        class_nodes
    ),
    "Class Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Subject {
        subject_id: row.subject_id
    })
    SET
        node.subject_name = row.subject_name,
        node.subject_category = row.subject_category,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        school_subject
    ),
    "Subject Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Teacher {
        teacher_id: row.teacher_id
    })
    SET
        node.teacher_name = row.teacher_name,
        node.first_name = row.first_name,
        node.last_name = row.last_name,
        node.teacher_status = row.teacher_status,
        node.subject_count = row.subject_count,
        node.class_count = row.class_count,
        node.primary_subject = row.primary_subject,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        teacher_nodes
    ),
    "Teacher Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    SET
        node.class_id = row.class_id,
        node.subject_id = row.subject_id,
        node.term_id = row.term_id,
        node.subject_name = row.subject_name,
        node.class_name = row.class_name,
        node.academic_group = row.academic_group,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        class_subject_nodes
    ),
    "Class Subject Nodes",
)

Writing Class Nodes : 18
Class Nodes Written: 18 Of 18
Writing Subject Nodes : 35
Subject Nodes Written: 35 Of 35
Writing Teacher Nodes : 45
Teacher Nodes Written: 45 Of 45
Writing Class Subject Nodes : 236
Class Subject Nodes Written: 236 Of 236


In [28]:
run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:MonthlySummary {
        monthly_summary_id: row.monthly_summary_id
    })
    SET
        node.enrollment_id = row.enrollment_id,
        node.student_id = row.student_id,
        node.class_id = row.class_id,
        node.month_name = row.month_name,
        node.attendance_percentage = row.attendance_percentage,
        node.average_score = row.average_score,
        node.subjects_failed = row.subjects_failed,
        node.performance_change = row.performance_change,
        node.summary_status = row.summary_status,
        node.risk_profile_id = row.risk_profile_id,
        node.risk_level = row.risk_level,
        node.risk_score = row.risk_score,
        node.risk_reason = row.risk_reason,
        node.recommended_action = row.recommended_action,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        monthly_nodes
    ),
    "Monthly Summary Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:SubjectRisk {
        subject_risk_id: row.subject_risk_id
    })
    SET
        node.monthly_summary_id = row.monthly_summary_id,
        node.student_id = row.student_id,
        node.class_subject_id = row.class_subject_id,
        node.subject_id = row.subject_id,
        node.subject_name = row.subject_name,
        node.subject_category = row.subject_category,
        node.month_name = row.month_name,
        node.subject_average = row.subject_average,
        node.class_subject_average = row.class_subject_average,
        node.performance_gap = row.performance_gap,
        node.risk_level = row.risk_level,
        node.risk_reason = row.risk_reason,
        node.recommended_action = row.recommended_action,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        subject_risk_nodes
    ),
    "Subject Risk Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Guardian {
        guardian_id: row.guardian_id
    })
    SET
        node.guardian_name = row.guardian_name,
        node.relationship_to_student = row.relationship_to_student,
        node.contact_masked = row.contact_masked,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        guardian_nodes
    ),
    "Guardian Nodes",
)

run_batch(
    '''
    UNWIND $rows AS row
    MERGE (node:Intervention {
        intervention_id: row.intervention_id
    })
    SET
        node.intervention_type = row.intervention_type,
        node.action_taken = row.action_taken,
        node.intervention_status = row.intervention_status,
        node.created_at = row.created_at,
        node.month_name = row.month_name,
        node.subject_name = row.subject_name,
        node.risk_reason = row.risk_reason,
        node.name = row.name,
        node.display_name = row.display_name,
        node.display_type = row.display_type,
        node.display_detail = row.display_detail,
        node.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        intervention_nodes
    ),
    "Intervention Nodes",
)

Writing Monthly Summary Nodes : 864
Monthly Summary Nodes Written: 864 Of 864
Writing Subject Risk Nodes : 11328
Subject Risk Nodes Written: 1000 Of 11328
Subject Risk Nodes Written: 2000 Of 11328
Subject Risk Nodes Written: 3000 Of 11328
Subject Risk Nodes Written: 4000 Of 11328
Subject Risk Nodes Written: 5000 Of 11328
Subject Risk Nodes Written: 6000 Of 11328
Subject Risk Nodes Written: 7000 Of 11328
Subject Risk Nodes Written: 8000 Of 11328
Subject Risk Nodes Written: 9000 Of 11328
Subject Risk Nodes Written: 10000 Of 11328
Subject Risk Nodes Written: 11000 Of 11328
Subject Risk Nodes Written: 11328 Of 11328
Writing Guardian Nodes : 216
Guardian Nodes Written: 216 Of 216
Writing Intervention Nodes : 253
Intervention Nodes Written: 253 Of 253


In [29]:
run_batch(
    '''
    UNWIND $rows AS row
    MATCH (student:Student {
        student_id: row.student_id
    })
    MATCH (class:Class {
        class_id: row.class_id
    })
    MERGE (student)-[relationship:ENROLLED_IN]->(class)
    SET
        relationship.enrollment_id = row.enrollment_id,
        relationship.term_id = row.term_id,
        relationship.enrollment_date = row.enrollment_date,
        relationship.enrollment_status = row.enrollment_status,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        enrollment_edges
    ),
    "Enrollment Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (class:Class {
        class_id: row.class_id
    })
    MATCH (class_subject:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    MERGE (class)-[relationship:HAS_CLASS_SUBJECT]->(class_subject)
    SET relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        class_subject
    ),
    "Class Subject Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (class_subject:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    MATCH (subject:Subject {
        subject_id: row.subject_id
    })
    MERGE (class_subject)-[relationship:FOR_SUBJECT]->(subject)
    SET relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        class_subject
    ),
    "Subject Assignment Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (teacher:Teacher {
        teacher_id: row.teacher_id
    })
    MATCH (class_subject:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    MERGE (teacher)-[relationship:TEACHES]->(class_subject)
    SET
        relationship.teacher_assignment_id = row.teacher_assignment_id,
        relationship.term_id = row.term_id,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        teacher_assignment
    ),
    "Teacher Assignment Relationships",
)

Writing Enrollment Relationships : 216
Enrollment Relationships Written: 216 Of 216
Writing Class Subject Relationships : 236
Class Subject Relationships Written: 236 Of 236
Writing Subject Assignment Relationships : 236
Subject Assignment Relationships Written: 236 Of 236
Writing Teacher Assignment Relationships : 236
Teacher Assignment Relationships Written: 236 Of 236


In [30]:
run_batch(
    '''
    UNWIND $rows AS row
    MATCH (student:Student {
        student_id: row.student_id
    })
    MATCH (summary:MonthlySummary {
        monthly_summary_id: row.monthly_summary_id
    })
    MERGE (student)-[relationship:HAS_MONTHLY_SUMMARY]->(summary)
    SET
        relationship.month_name = row.month_name,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        monthly_nodes
    ),
    "Monthly Summary Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (student:Student {
        student_id: row.student_id
    })
    MATCH (risk:SubjectRisk {
        subject_risk_id: row.subject_risk_id
    })
    MERGE (student)-[relationship:HAS_SUBJECT_RISK]->(risk)
    SET
        relationship.month_name = row.month_name,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        subject_risk_nodes
    ),
    "Student Subject Risk Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (risk:SubjectRisk {
        subject_risk_id: row.subject_risk_id
    })
    MATCH (class_subject:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    MERGE (risk)-[relationship:FOR_CLASS_SUBJECT]->(class_subject)
    SET relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        subject_risk_nodes
    ),
    "Risk Class Subject Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (student:Student {
        student_id: row.student_id
    })
    MATCH (class_subject:ClassSubject {
        class_subject_id: row.class_subject_id
    })
    MERGE (student)-[relationship:PERFORMED]->(class_subject)
    SET
        relationship.month_name = "April",
        relationship.subject_average = row.subject_average,
        relationship.risk_level = row.risk_level,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        april_performance
    ),
    "April Performance Relationships",
)

Writing Monthly Summary Relationships : 864
Monthly Summary Relationships Written: 864 Of 864
Writing Student Subject Risk Relationships : 11328
Student Subject Risk Relationships Written: 1000 Of 11328
Student Subject Risk Relationships Written: 2000 Of 11328
Student Subject Risk Relationships Written: 3000 Of 11328
Student Subject Risk Relationships Written: 4000 Of 11328
Student Subject Risk Relationships Written: 5000 Of 11328
Student Subject Risk Relationships Written: 6000 Of 11328
Student Subject Risk Relationships Written: 7000 Of 11328
Student Subject Risk Relationships Written: 8000 Of 11328
Student Subject Risk Relationships Written: 9000 Of 11328
Student Subject Risk Relationships Written: 10000 Of 11328
Student Subject Risk Relationships Written: 11000 Of 11328
Student Subject Risk Relationships Written: 11328 Of 11328
Writing Risk Class Subject Relationships : 11328
Risk Class Subject Relationships Written: 1000 Of 11328
Risk Class Subject Relationships Written: 2000 Of 1

In [31]:
run_batch(
    '''
    UNWIND $rows AS row
    MATCH (guardian:Guardian {
        guardian_id: row.guardian_id
    })
    MATCH (student:Student {
        student_id: row.student_id
    })
    MERGE (guardian)-[relationship:GUARDIAN_OF]->(student)
    SET
        relationship.relationship_to_student = row.relationship_to_student,
        relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        guardian_nodes
    ),
    "Guardian Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (student:Student {
        student_id: row.student_id
    })
    MATCH (intervention:Intervention {
        intervention_id: row.intervention_id
    })
    MERGE (student)-[relationship:RECEIVED_INTERVENTION]->(intervention)
    SET relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        intervention_nodes
    ),
    "Student Intervention Relationships",
)

run_batch(
    '''
    UNWIND $rows AS row
    MATCH (intervention:Intervention {
        intervention_id: row.intervention_id
    })
    MATCH (risk:SubjectRisk {
        subject_risk_id: row.subject_risk_id
    })
    MERGE (intervention)-[relationship:ADDRESSES]->(risk)
    SET relationship.pipeline_run_id = $pipeline_run_id
    ''',
    clean_records(
        intervention_nodes
    ),
    "Intervention Risk Relationships",
)

Writing Guardian Relationships : 216
Guardian Relationships Written: 216 Of 216
Writing Student Intervention Relationships : 253
Student Intervention Relationships Written: 253 Of 253
Writing Intervention Risk Relationships : 253
Intervention Risk Relationships Written: 253 Of 253


In [32]:
def short_subject_expression(alias):
    return f'''
    CASE
        WHEN {alias}.subject_name = "Agricultural Science"
            THEN "Agric Science"
        WHEN {alias}.subject_name = "Further Mathematics"
            THEN "Further Maths"
        WHEN {alias}.subject_name = "Christian Religious Studies"
            THEN "CRS"
        WHEN {alias}.subject_name = "Islamic Religious Studies"
            THEN "IRS"
        WHEN {alias}.subject_name = "Physical and Health Education"
            THEN "PHE"
        WHEN {alias}.subject_name = "Information and Communication Technology"
            THEN "ICT"
        WHEN size({alias}.subject_name) <= 16
            THEN {alias}.subject_name
        ELSE substring(
            {alias}.subject_name,
            0,
            15
        )
    END
    '''


run_query(
    '''
    MATCH (student:Student)
    SET
        student.full_display_name = student.student_name,
        student.name =
            CASE
                WHEN size(student.student_name) <= 16
                    THEN student.student_name
                ELSE
                    student.first_name
                    + " "
                    + substring(
                        student.last_name,
                        0,
                        1
                    )
                    + "."
            END,
        student.display_name = student.name,
        student.graph_caption = student.name
    '''
)

run_query(
    '''
    MATCH (guardian:Guardian)
    SET
        guardian.full_display_name = guardian.guardian_name,
        guardian.name = "Guardian",
        guardian.display_name = "Guardian",
        guardian.graph_caption = "Guardian"
    '''
)

run_query(
    '''
    MATCH (class:Class)
    SET
        class.full_display_name =
            class.class_name
            + " | "
            + class.academic_group,
        class.name = class.class_name,
        class.display_name = class.class_name,
        class.graph_caption = class.class_name
    '''
)

run_query(
    '''
    MATCH (teacher:Teacher)
    SET
        teacher.full_display_name = teacher.teacher_name,
        teacher.name =
            CASE
                WHEN size(teacher.teacher_name) <= 16
                    THEN teacher.teacher_name
                ELSE
                    teacher.first_name
                    + " "
                    + substring(
                        teacher.last_name,
                        0,
                        1
                    )
                    + "."
            END,
        teacher.display_name = teacher.name,
        teacher.graph_caption = teacher.name
    '''
)

run_query(
    '''
    MATCH (subject:Subject)
    SET
        subject.full_display_name = subject.subject_name,
        subject.name =
            CASE
                WHEN subject.subject_name = "Agricultural Science"
                    THEN "Agric Science"
                WHEN subject.subject_name = "Further Mathematics"
                    THEN "Further Maths"
                WHEN subject.subject_name = "Christian Religious Studies"
                    THEN "CRS"
                WHEN subject.subject_name = "Islamic Religious Studies"
                    THEN "IRS"
                WHEN subject.subject_name = "Physical and Health Education"
                    THEN "PHE"
                WHEN subject.subject_name = "Information and Communication Technology"
                    THEN "ICT"
                WHEN size(subject.subject_name) <= 16
                    THEN subject.subject_name
                ELSE substring(
                    subject.subject_name,
                    0,
                    15
                )
            END,
        subject.display_name = subject.name,
        subject.graph_caption = subject.name
    '''
)

run_query(
    '''
    MATCH (class_subject:ClassSubject)
    SET
        class_subject.full_display_name =
            class_subject.subject_name
            + " | "
            + class_subject.class_name,
        class_subject.name =
            CASE
                WHEN class_subject.subject_name = "Agricultural Science"
                    THEN "Agric Sci"
                WHEN class_subject.subject_name = "Further Mathematics"
                    THEN "Further Math"
                WHEN class_subject.subject_name = "Christian Religious Studies"
                    THEN "CRS"
                WHEN class_subject.subject_name = "Islamic Religious Studies"
                    THEN "IRS"
                WHEN class_subject.subject_name = "Physical and Health Education"
                    THEN "PHE"
                WHEN class_subject.subject_name = "Information and Communication Technology"
                    THEN "ICT"
                WHEN size(class_subject.subject_name) <= 10
                    THEN class_subject.subject_name
                ELSE substring(
                    class_subject.subject_name,
                    0,
                    10
                )
            END
            + " | "
            + class_subject.class_name
    '''
)

run_query(
    '''
    MATCH (summary:MonthlySummary)
    SET
        summary.full_display_name =
            summary.month_name
            + " Summary | "
            + summary.risk_level
            + " Risk",
        summary.name =
            summary.month_name
            + " | "
            + CASE
                WHEN summary.risk_level = "Medium"
                    THEN "Med"
                ELSE summary.risk_level
            END
    '''
)

run_query(
    '''
    MATCH (risk:SubjectRisk)
    SET
        risk.full_display_name =
            risk.subject_name
            + " | "
            + risk.month_name
            + " | "
            + risk.risk_level
            + " Risk",
        risk.name =
            CASE
                WHEN risk.subject_name = "Agricultural Science"
                    THEN "Agric Sci"
                WHEN risk.subject_name = "Further Mathematics"
                    THEN "Further Math"
                WHEN risk.subject_name = "Christian Religious Studies"
                    THEN "CRS"
                WHEN risk.subject_name = "Islamic Religious Studies"
                    THEN "IRS"
                WHEN risk.subject_name = "Physical and Health Education"
                    THEN "PHE"
                WHEN risk.subject_name = "Information and Communication Technology"
                    THEN "ICT"
                WHEN size(risk.subject_name) <= 10
                    THEN risk.subject_name
                ELSE substring(
                    risk.subject_name,
                    0,
                    10
                )
            END
            + " | "
            + CASE
                WHEN risk.risk_level = "Medium"
                    THEN "Med"
                ELSE risk.risk_level
            END
    '''
)

run_query(
    '''
    MATCH (intervention:Intervention)
    SET
        intervention.full_display_name =
            intervention.intervention_type
            + " | "
            + intervention.subject_name
            + " | "
            + intervention.month_name,
        intervention.name =
            CASE
                WHEN intervention.subject_name = "Agricultural Science"
                    THEN "Agric Support"
                WHEN intervention.subject_name = "Further Mathematics"
                    THEN "Math Support"
                WHEN intervention.subject_name = "Christian Religious Studies"
                    THEN "CRS Support"
                WHEN intervention.subject_name = "Islamic Religious Studies"
                    THEN "IRS Support"
                WHEN intervention.subject_name = "Physical and Health Education"
                    THEN "PHE Support"
                WHEN intervention.subject_name = "Information and Communication Technology"
                    THEN "ICT Support"
                WHEN size(intervention.subject_name) <= 10
                    THEN intervention.subject_name + " Support"
                ELSE substring(
                    intervention.subject_name,
                    0,
                    9
                ) + " Support"
            END
    '''
)

run_query(
    '''
    MATCH (node)
    WHERE node.name IS NOT NULL
    SET node.name =
        CASE
            WHEN size(node.name) <= 20
                THEN node.name
            ELSE substring(
                node.name,
                0,
                20
            )
        END
    '''
)

run_query(
    '''
    MATCH (node)
    WHERE node.name IS NOT NULL
    SET
        node.display_name = node.name,
        node.graph_caption = node.name
    '''
)

caption_validation = pd.DataFrame(
    run_query(
        '''
        MATCH (node)
        UNWIND labels(node) AS node_label
        RETURN
            node_label AS node_label,
            count(*) AS total_nodes,
            count(
                CASE
                    WHEN
                        node.graph_caption IS NOT NULL
                        AND size(node.graph_caption) <= 20
                    THEN 1
                END
            ) AS concise_caption_nodes,
            max(
                size(node.graph_caption)
            ) AS longest_caption_length
        ORDER BY total_nodes DESC
        '''
    )
)

caption_validation[
    "validation_status"
] = np.where(
    caption_validation[
        "total_nodes"
    ] == caption_validation[
        "concise_caption_nodes"
    ],
    "PASS",
    "CHECK",
)

show_table(
    "Concise Graph Caption Validation",
    caption_validation,
    rows=len(
        caption_validation
    ),
)

total_graph_nodes = int(
    caption_validation[
        "total_nodes"
    ].sum()
)

total_concise_captions = int(
    caption_validation[
        "concise_caption_nodes"
    ].sum()
)

if total_concise_captions != total_graph_nodes:
    raise RuntimeError(
        "Not every Neo4j node has a concise graph caption. "
        "Expected "
        + str(total_graph_nodes)
        + " but found "
        + str(total_concise_captions)
        + "."
    )

print(
    "All",
    total_concise_captions,
    "Neo4j Nodes Have Concise Recruiter Friendly Captions",
    flush=True,
)

Concise Graph Caption Validation


,Node Label,Total Nodes,Concise Caption Nodes,Longest Caption Length,Validation Status
0,SubjectRisk,11328,11328,19,PASS
1,MonthlySummary,864,864,15,PASS
2,Intervention,253,253,18,PASS
3,ClassSubject,236,236,19,PASS
4,Student,216,216,16,PASS
5,Guardian,216,216,8,PASS
6,Teacher,45,45,16,PASS
7,Subject,35,35,16,PASS
8,Class,18,18,5,PASS


All 13211 Neo4j Nodes Have Concise Recruiter Friendly Captions


In [33]:
node_count_data = pd.DataFrame(
    run_query(
        '''
        MATCH (node)
        UNWIND labels(node) AS label
        RETURN
            label AS node_label,
            count(*) AS node_count
        ORDER BY node_count DESC
        '''
    )
)

relationship_count_data = pd.DataFrame(
    run_query(
        '''
        MATCH ()-[relationship]->()
        RETURN
            type(relationship) AS relationship_type,
            count(*) AS relationship_count
        ORDER BY relationship_count DESC
        '''
    )
)

show_table(
    "Neo4j Node Validation",
    node_count_data,
    rows=len(node_count_data),
)

show_table(
    "Neo4j Relationship Validation",
    relationship_count_data,
    rows=len(relationship_count_data),
)

expected_node_counts = {
    "Student": len(student_nodes),
    "Class": len(class_nodes),
    "Subject": len(school_subject),
    "Teacher": len(teacher_nodes),
    "ClassSubject": len(class_subject_nodes),
    "MonthlySummary": len(monthly_nodes),
    "SubjectRisk": len(subject_risk_nodes),
    "Guardian": len(guardian_nodes),
    "Intervention": len(intervention_nodes),
}

actual_node_counts = dict(
    zip(
        node_count_data[
            "node_label"
        ],
        node_count_data[
            "node_count"
        ],
    )
)

validation_rows = []

for label_name, expected_count in expected_node_counts.items():
    actual_count = int(
        actual_node_counts.get(
            label_name,
            0,
        )
    )

    validation_rows.append(
        {
            "check_name": (
                label_name
                + " Node Count"
            ),
            "actual_value": actual_count,
            "expected_value": expected_count,
            "validation_status": (
                "PASS"
                if actual_count
                == expected_count
                else "CHECK"
            ),
        }
    )

validation_report = pd.DataFrame(
    validation_rows
)

show_table(
    "Neo4j Count Validation",
    validation_report,
    rows=len(validation_report),
)

if not (
    validation_report[
        "validation_status"
    ] == "PASS"
).all():
    raise RuntimeError(
        "Neo4j Node Count Validation Failed"
    )

Neo4j Node Validation


,Node Label,Node Count
0,SubjectRisk,11328
1,MonthlySummary,864
2,Intervention,253
3,ClassSubject,236
4,Student,216
5,Guardian,216
6,Teacher,45
7,Subject,35
8,Class,18


Neo4j Relationship Validation


,Relationship Type,Relationship Count
0,HAS_SUBJECT_RISK,11328
1,FOR_CLASS_SUBJECT,11328
2,PERFORMED,2832
3,HAS_MONTHLY_SUMMARY,864
4,RECEIVED_INTERVENTION,253
5,ADDRESSES,253
6,HAS_CLASS_SUBJECT,236
7,TEACHES,236
8,FOR_SUBJECT,236
9,ENROLLED_IN,216


Neo4j Count Validation


,Check Name,Actual Value,Expected Value,Validation Status
0,Student Node Count,216,216,PASS
1,Class Node Count,18,18,PASS
2,Subject Node Count,35,35,PASS
3,Teacher Node Count,45,45,PASS
4,ClassSubject Node Count,236,236,PASS
5,MonthlySummary Node Count,864,864,PASS
6,SubjectRisk Node Count,11328,11328,PASS
7,Guardian Node Count,216,216,PASS
8,Intervention Node Count,253,253,PASS


In [34]:
high_risk_student_insight = pd.DataFrame(
    run_query(
        '''
        MATCH
            (student:Student)
            -[:HAS_MONTHLY_SUMMARY]->
            (summary:MonthlySummary)
        WHERE summary.risk_level = "High"
        RETURN
            student.student_id AS student_id,
            student.student_name AS student_name,
            count(summary) AS high_risk_months,
            round(
                avg(summary.average_score),
                2
            ) AS average_score,
            round(
                avg(summary.attendance_percentage),
                2
            ) AS average_attendance
        ORDER BY
            high_risk_months DESC,
            average_score ASC
        LIMIT 20
        '''
    )
)

teacher_risk_exposure = pd.DataFrame(
    run_query(
        '''
        MATCH
            (teacher:Teacher)
            -[:TEACHES]->
            (class_subject:ClassSubject)
            <-[:FOR_CLASS_SUBJECT]-
            (risk:SubjectRisk)
        WHERE risk.risk_level IN [
            "High",
            "Medium"
        ]
        RETURN
            teacher.teacher_id AS teacher_id,
            teacher.teacher_name AS teacher_name,
            count(risk) AS risk_records,
            count(
                DISTINCT risk.student_id
            ) AS students_requiring_support
        ORDER BY
            risk_records DESC,
            students_requiring_support DESC
        LIMIT 20
        '''
    )
)

class_risk_concentration = pd.DataFrame(
    run_query(
        '''
        MATCH
            (student:Student)
            -[:ENROLLED_IN]->
            (class:Class),
            (student)
            -[:HAS_MONTHLY_SUMMARY]->
            (summary:MonthlySummary)
        WHERE summary.risk_level IN [
            "High",
            "Medium"
        ]
        RETURN
            class.class_name AS class_name,
            class.academic_group AS academic_group,
            count(summary) AS risk_records,
            count(
                DISTINCT student
            ) AS students_at_risk
        ORDER BY
            risk_records DESC,
            students_at_risk DESC
        '''
    )
)

show_table(
    "Students Requiring The Most Support",
    high_risk_student_insight,
    rows=20,
)

show_table(
    "Teacher Risk Exposure",
    teacher_risk_exposure,
    rows=20,
)

show_table(
    "Class Risk Concentration",
    class_risk_concentration,
    rows=len(class_risk_concentration),
)

Students Requiring The Most Support


,Student ID,Student Name,High Risk Months,Average Score,Average Attendance
0,STU0069,Deborah Okeke,4,42.10,66.65
1,STU0117,Grace Olaniyan,4,42.18,64.32
2,STU0140,Isaac Danjuma,4,42.83,61.74
3,STU0005,Olamide Asuquo,4,43.22,67.95
4,STU0024,Kelechi Ojo,4,43.25,57.29
5,STU0016,Stephen Uche,4,43.58,68.96
6,STU0090,Kelechi Adeyemi,4,43.91,74.58
7,STU0134,Daniel Adelaja,4,44.03,60.85
8,STU0145,Henry Udoh,4,44.10,74.53
9,STU0020,Oluwaseun Asuquo,4,46.46,67.20


Teacher Risk Exposure


,Teacher ID,Teacher Name,Risk Records,Students Requiring Support
0,TCH001,Adekunle Eze,354,109
1,TCH010,Blessing Bello,286,70
2,TCH003,Yusuf Ekanem,282,85
3,TCH005,Gabriel Olawale,273,85
4,TCH008,Lilian Oladiran,224,68
5,TCH012,Amaka Mohammed,221,66
6,TCH007,Simon Omoregie,153,45
7,TCH037,Victor Jatau,148,48
8,TCH031,Rotimi Musa,148,43
9,TCH026,Rahmat Ezeh,147,42


Class Risk Concentration


,Class Name,Academic Group,Risk Records,Students At Risk
0,SS2D,Arts,32,12
1,SS1D,Arts,31,10
2,SS1C,Commercial,30,11
3,JSS3B,Junior Secondary,30,10
4,SS2B,Science,29,10
5,JSS1B,Junior Secondary,29,8
6,JSS2A,Junior Secondary,28,11
7,SS1A,Science,27,12
8,SS1B,Science,27,10
9,SS3A,Science,27,10


In [35]:
display_name_validation = pd.DataFrame(
    run_query(
        '''
        MATCH (node)
        WHERE
            node.name IS NOT NULL
            AND node.display_name IS NOT NULL
            AND node.display_type IS NOT NULL
        UNWIND labels(node) AS node_label
        RETURN
            node_label AS node_label,
            count(*) AS readable_nodes
        ORDER BY readable_nodes DESC
        '''
    )
)

show_table(
    "Recruiter Display Name Validation",
    display_name_validation,
    rows=len(
        display_name_validation
    ),
)

representative_student = run_query(
    '''
    MATCH
        (student:Student)
        -[:RECEIVED_INTERVENTION]->
        (:Intervention)
    RETURN
        student.student_id AS student_id,
        student.student_name AS student_name
    ORDER BY student.student_id
    LIMIT 1
    '''
)

if representative_student:
    portfolio_student_id = representative_student[
        0
    ]["student_id"]
else:
    portfolio_student_id = student_nodes.iloc[
        0
    ]["student_id"]

representative_class = run_query(
    '''
    MATCH
        (student:Student {
            student_id: $student_id
        })
        -[:ENROLLED_IN]->
        (class:Class)
    RETURN
        class.class_id AS class_id,
        class.class_name AS class_name
    LIMIT 1
    ''',
    {
        "student_id": portfolio_student_id,
    },
)

if representative_class:
    portfolio_class_id = representative_class[
        0
    ]["class_id"]
else:
    portfolio_class_id = class_nodes.iloc[
        0
    ]["class_id"]

student_snapshot_query = f'''
MATCH path =
    (student:Student {{
        student_id: "{portfolio_student_id}"
    }})
    -[*1..2]-
    (connected)
WHERE
    connected:Class
    OR connected:Guardian
    OR (
        connected:MonthlySummary
        AND connected.month_name = "April"
    )
    OR (
        connected:SubjectRisk
        AND connected.month_name = "April"
        AND connected.risk_level IN [
            "High",
            "Medium"
        ]
    )
    OR connected:Intervention
RETURN path
LIMIT 35
'''

class_risk_query = f'''
MATCH path =
    (class:Class {{
        class_id: "{portfolio_class_id}"
    }})
    <-[:ENROLLED_IN]-
    (student:Student)
    -[:HAS_MONTHLY_SUMMARY]->
    (summary:MonthlySummary {{
        month_name: "April"
    }})
WHERE summary.risk_level IN [
    "High",
    "Medium"
]
RETURN path
LIMIT 30
'''

intervention_story_query = '''
MATCH path =
    (student:Student)
    -[:RECEIVED_INTERVENTION]->
    (intervention:Intervention)
    -[:ADDRESSES]->
    (risk:SubjectRisk)
    -[:FOR_CLASS_SUBJECT]->
    (class_subject:ClassSubject)
    -[:FOR_SUBJECT]->
    (subject:Subject)
RETURN path
LIMIT 20
'''

teacher_support_query = '''
MATCH path =
    (teacher:Teacher)
    -[:TEACHES]->
    (class_subject:ClassSubject)
    <-[:FOR_CLASS_SUBJECT]-
    (risk:SubjectRisk {
        month_name: "April",
        risk_level: "High"
    })
RETURN path
LIMIT 30
'''

recruiter_query_file = (
    output_directory
    / "SchoolPulse Recruiter Queries.cypher"
)

recruiter_query_file.write_text(
    (
        "// STUDENT SNAPSHOT\n"
        + student_snapshot_query.strip()
        + ";\n\n"
        + "// CLASS RISK SNAPSHOT\n"
        + class_risk_query.strip()
        + ";\n\n"
        + "// INTERVENTION STORY\n"
        + intervention_story_query.strip()
        + ";\n\n"
        + "// TEACHER SUPPORT VIEW\n"
        + teacher_support_query.strip()
        + ";\n"
    ),
    encoding="utf-8",
)

recruiter_guide = pd.DataFrame(
    [
        {
            "presentation_view": (
                "Student Snapshot"
            ),
            "purpose": (
                "Shows one student, class, guardian, "
                "April performance, risks and interventions"
            ),
            "recommended_use": (
                "Best opening graph for recruiters"
            ),
        },
        {
            "presentation_view": (
                "Class Risk Snapshot"
            ),
            "purpose": (
                "Shows students requiring support "
                "inside one class"
            ),
            "recommended_use": (
                "Demonstrates class level risk monitoring"
            ),
        },
        {
            "presentation_view": (
                "Intervention Story"
            ),
            "purpose": (
                "Shows the complete path from student "
                "to intervention to subject risk"
            ),
            "recommended_use": (
                "Demonstrates how the system supports action"
            ),
        },
        {
            "presentation_view": (
                "Teacher Support View"
            ),
            "purpose": (
                "Shows teachers connected to high risk "
                "subject areas"
            ),
            "recommended_use": (
                "Demonstrates teacher support planning"
            ),
        },
    ]
)

show_table(
    "Recruiter Presentation Views",
    recruiter_guide,
    rows=len(
        recruiter_guide
    ),
)

print(
    "Recruiter Queries Saved To:",
    recruiter_query_file,
)

print(
    "Recommended Student Snapshot ID:",
    portfolio_student_id,
)

print(
    "Recommended Class Snapshot ID:",
    portfolio_class_id,
)

Recruiter Display Name Validation


,Node Label,Readable Nodes
0,SubjectRisk,11328
1,MonthlySummary,864
2,Intervention,253
3,ClassSubject,236
4,Student,216
5,Guardian,216
6,Teacher,45
7,Subject,35
8,Class,18


Recruiter Presentation Views


,Presentation View,Purpose,Recommended Use
0,Student Snapshot,"Shows one student, class, guardian, April perf...",Best opening graph for recruiters
1,Class Risk Snapshot,Shows students requiring support inside one class,Demonstrates class level risk monitoring
2,Intervention Story,Shows the complete path from student to interv...,Demonstrates how the system supports action
3,Teacher Support View,Shows teachers connected to high risk subject ...,Demonstrates teacher support planning


Recruiter Queries Saved To: <USER_HOME>\SchoolPulse Portfolio Outputs\SchoolPulse Recruiter Queries.cypher
Recommended Student Snapshot ID: STU0002
Recommended Class Snapshot ID: JC1A


In [36]:
report = {
    "Pipeline": pipeline_name,
    "Pipeline Run ID": pipeline_run_id,
    "Neo4j URI": neo4j_uri,
    "Neo4j Database": neo4j_database,
    "Students": len(student_nodes),
    "Classes": len(class_nodes),
    "Subjects": len(school_subject),
    "Teachers": len(teacher_nodes),
    "Class Subjects": len(class_subject),
    "Monthly Summaries": len(monthly_nodes),
    "Subject Risks": len(subject_risk_nodes),
    "Guardians": len(guardian_nodes),
    "Interventions": len(intervention_nodes),
    "Readable Node Captions": int(
        display_name_validation[
            "readable_nodes"
        ].sum()
    ),
    "Concise Caption Labels": (
        total_concise_captions
    ),
    "Total Graph Nodes": (
        total_graph_nodes
    ),
    "Concise Caption Validation": (
        "PASS"
        if total_concise_captions
        == total_graph_nodes
        else "CHECK"
    ),
    "Recruiter Student Snapshot ID": (
        portfolio_student_id
    ),
    "Recruiter Class Snapshot ID": (
        portfolio_class_id
    ),
    "Recruiter Presentation Status": "PASS",
    "Validation Status": "PASS",
}

report_path = (
    output_directory
    / "Neo4j Pipeline Report.json"
)

with report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        report,
        file,
        indent=4,
    )

node_count_data.to_csv(
    output_directory
    / "Neo4j Node Counts.csv",
    index=False,
)

relationship_count_data.to_csv(
    output_directory
    / "Neo4j Relationship Counts.csv",
    index=False,
)

high_risk_student_insight.to_csv(
    output_directory
    / "Neo4j High Risk Students.csv",
    index=False,
)

teacher_risk_exposure.to_csv(
    output_directory
    / "Neo4j Teacher Risk Exposure.csv",
    index=False,
)

class_risk_concentration.to_csv(
    output_directory
    / "Neo4j Class Risk Concentration.csv",
    index=False,
)

display(
    pd.DataFrame(
        [
            {
                "Check Name": key,
                "Result": value,
            }
            for key, value
            in report.items()
        ]
    )
)

print(
    "Neo4j Pipeline Report Saved To:",
    report_path,
)

print(
    "SchoolPulse Neo4j Pipeline And Recruiter Presentation "
    "Completed Successfully"
)

driver.close()

,Check Name,Result
0,Pipeline,SchoolPulse Neo4j Pipeline Version 2
1,Pipeline Run ID,RUN20260718T043558Z
2,Neo4j URI,bolt://localhost:7687
3,Neo4j Database,schoolpulse
4,Students,216
5,Classes,18
6,Subjects,35
7,Teachers,45
8,Class Subjects,236
9,Monthly Summaries,864


Neo4j Pipeline Report Saved To: <USER_HOME>\SchoolPulse Portfolio Outputs\Neo4j Pipeline Report.json
SchoolPulse Neo4j Pipeline And Recruiter Presentation Completed Successfully
